In [2]:
"""
Payer Network Data - Excel to SQLite Database Loader

This script loads data from Sample_Payer_Network_Data.xlsx and creates
a SQLite database with properly structured tables.

Usage:
    python load_payer_data.py

Requirements:
    - pandas
    - openpyxl (for Excel file reading)
"""

import pandas as pd
import sqlite3
import json
from pathlib import Path
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')


class PayerNetworkDataLoader:
    """Class to handle loading Payer Network Data into SQLite database"""
    
    def __init__(self, excel_file_path, database_path, schema_json_path=None):
        """
        Initialize the data loader
        
        Args:
            excel_file_path (str): Path to the Excel file
            database_path (str): Path where SQLite database will be created
            schema_json_path (str, optional): Path to schema JSON file
        """
        self.excel_file_path = excel_file_path
        self.database_path = database_path
        self.schema_json_path = schema_json_path
        self.dataframes = {}
        self.conn = None
        
        # Sheet to table mapping
        self.sheet_to_table_mapping = {
            "Providers": "providers",
            "Members": "members",
            "Referral_Network": "referral_network",
            "Monthly_Metrics": "monthly_metrics",
            "Provider_Pods": "provider_pods",
            "Behavior_Opportunities": "behavior_opportunities",
            "High_Leakage_Providers": "high_leakage_providers",
            "High_Risk_Members": "high_risk_members",
            "Data_Dictionary": "data_dictionary"
        }
    
    def standardize_column_names(self, df):
        """
        Standardize column names to snake_case
        
        Args:
            df (DataFrame): Input dataframe
            
        Returns:
            DataFrame: Dataframe with standardized column names
        """
        df.columns = df.columns.str.strip()
        df.columns = df.columns.str.replace("'", "'", regex=False)
        df.columns = df.columns.str.replace("-", "-", regex=False)
        df.columns = df.columns.str.replace(" ", "_", regex=False)
        df.columns = df.columns.str.replace("/", "_", regex=False)
        df.columns = df.columns.str.replace("-", "_", regex=False)
        df.columns = df.columns.str.lower()
        
        return df
    
    def load_schema(self):
        """Load schema from JSON file if provided"""
        if self.schema_json_path and Path(self.schema_json_path).exists():
            with open(self.schema_json_path, 'r') as f:
                schema = json.load(f)
            print(f"✅ Schema loaded from {self.schema_json_path}")
            return schema
        else:
            print("⚠️  No schema file provided or file not found")
            return None
    
    def load_excel_data(self):
        """Load all sheets from Excel file into dataframes"""
        print(f"\n{'='*80}")
        print("LOADING DATA FROM EXCEL FILE")
        print(f"{'='*80}\n")
        
        excel_file = pd.ExcelFile(self.excel_file_path)
        print(f"📊 Found {len(excel_file.sheet_names)} sheets in Excel file\n")
        
        for sheet_name in excel_file.sheet_names:
            try:
                # Read the sheet
                df = pd.read_excel(excel_file, sheet_name=sheet_name)
                
                # Standardize column names
                df = self.standardize_column_names(df)
                
                # Get table name
                table_name = self.sheet_to_table_mapping.get(
                    sheet_name, 
                    sheet_name.lower()
                )
                
                # Store in dictionary
                self.dataframes[table_name] = df
                
                print(f"✅ {sheet_name:30} → {table_name:30} "
                      f"({df.shape[0]:,} rows × {df.shape[1]} columns)")
                
            except Exception as e:
                print(f"❌ Error loading {sheet_name}: {e}")
        
        print(f"\n✅ Successfully loaded {len(self.dataframes)} tables\n")
        return self.dataframes
    
    def create_database(self):
        """Create SQLite database and insert all data"""
        print(f"\n{'='*80}")
        print("CREATING SQLITE DATABASE")
        print(f"{'='*80}\n")
        
        # Create database connection
        self.conn = sqlite3.connect(self.database_path)
        print(f"🔗 Connected to database: {self.database_path}\n")
        
        # Insert each dataframe as a table
        print("💾 Inserting data into database...\n")
        
        for table_name, df in self.dataframes.items():
            try:
                df.to_sql(table_name, self.conn, if_exists="replace", index=False)
                print(f"✅ {table_name:30} {len(df):,} rows inserted")
            except Exception as e:
                print(f"❌ Error inserting {table_name}: {e}")
        
        self.conn.commit()
        print(f"\n✅ All data successfully inserted into {self.database_path}\n")
    
    def verify_database(self):
        """Verify database creation and display summary"""
        print(f"\n{'='*80}")
        print("DATABASE VERIFICATION")
        print(f"{'='*80}\n")
        
        cursor = self.conn.cursor()
        
        # List all tables
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = cursor.fetchall()
        
        print("📋 Tables in database:\n")
        
        total_rows = 0
        for table in tables:
            table_name = table[0]
            cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
            count = cursor.fetchone()[0]
            total_rows += count
            print(f"  • {table_name:30} {count:,} rows")
        
        print(f"\n{'='*80}")
        print(f"Total Records: {total_rows:,}")
        print(f"Total Tables: {len(tables)}")
        print(f"Database File: {self.database_path}")
        print(f"{'='*80}\n")
    
    def display_table_schemas(self):
        """Display schema information for all tables"""
        print(f"\n{'='*80}")
        print("TABLE SCHEMAS")
        print(f"{'='*80}\n")
        
        cursor = self.conn.cursor()
        
        for table_name in self.dataframes.keys():
            print(f"\nTable: {table_name}")
            print("-" * 80)
            
            cursor.execute(f"PRAGMA table_info({table_name});")
            columns = cursor.fetchall()
            
            print(f"Columns ({len(columns)} total):")
            for col in columns:
                col_id, col_name, col_type, not_null, default_val, pk = col
                pk_marker = " [PRIMARY KEY]" if pk else ""
                print(f"  • {col_name:40} {col_type}{pk_marker}")
    
    def run_sample_queries(self):
        """Run sample queries to test the database"""
        print(f"\n{'='*80}")
        print("SAMPLE QUERIES")
        print(f"{'='*80}\n")
        
        queries = [
            {
                "name": "Providers by Specialty",
                "sql": """
                    SELECT specialty, COUNT(*) as provider_count
                    FROM providers
                    GROUP BY specialty
                    ORDER BY provider_count DESC
                """
            },
            {
                "name": "Members by Risk Level",
                "sql": """
                    SELECT risk_level, COUNT(*) as member_count,
                           ROUND(AVG(annual_medical_cost), 2) as avg_cost
                    FROM members
                    GROUP BY risk_level
                    ORDER BY member_count DESC
                """
            },
            {
                "name": "Top 10 High Leakage Providers",
                "sql": """
                    SELECT provider_name, specialty, 
                           ROUND(estimated_annual_revenue_loss, 2) as revenue_loss
                    FROM high_leakage_providers
                    ORDER BY estimated_annual_revenue_loss DESC
                    LIMIT 10
                """
            }
        ]
        
        for query_info in queries:
            print(f"\n📊 {query_info['name']}:")
            print("-" * 80)
            try:
                result = pd.read_sql_query(query_info['sql'], self.conn)
                print(result.to_string(index=False))
            except Exception as e:
                print(f"❌ Error: {e}")
    
    def close(self):
        """Close database connection"""
        if self.conn:
            self.conn.close()
            print("\n🔒 Database connection closed")
    
    def run(self, show_schemas=False, run_queries=False):
        """
        Run the complete data loading process
        
        Args:
            show_schemas (bool): Whether to display table schemas
            run_queries (bool): Whether to run sample queries
        """
        start_time = datetime.now()
        
        print("\n" + "="*80)
        print("PAYER NETWORK DATA LOADER")
        print("="*80)
        print(f"Started at: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
        
        try:
            # Load schema if available
            self.load_schema()
            
            # Load Excel data
            self.load_excel_data()
            
            # Create database
            self.create_database()
            
            # Verify database
            self.verify_database()
            
            # Optional: Display schemas
            if show_schemas:
                self.display_table_schemas()
            
            # Optional: Run sample queries
            if run_queries:
                self.run_sample_queries()
            
            end_time = datetime.now()
            duration = (end_time - start_time).total_seconds()
            
            print(f"\n{'='*80}")
            print("✅ SUCCESS!")
            print(f"{'='*80}")
            print(f"Completed in: {duration:.2f} seconds")
            print(f"Database created: {self.database_path}")
            print(f"{'='*80}\n")
            
        except Exception as e:
            print(f"\n❌ ERROR: {e}")
            raise
        
        finally:
            self.close()


def main():
    """Main function to run the data loader"""
    
    # Configuration - Update these paths as needed
    EXCEL_FILE = "Sample_Payer_Network_Data.xlsx"
    DATABASE_FILE = "my_database.db"
    SCHEMA_FILE = "payer_network_data_schema.json"
    
    # Create loader instance
    loader = PayerNetworkDataLoader(
        excel_file_path=EXCEL_FILE,
        database_path=DATABASE_FILE,
        schema_json_path=SCHEMA_FILE
    )
    
    # Run the loading process
    loader.run(
        show_schemas=True,   # Set to True to display table schemas
        run_queries=True     # Set to True to run sample queries
    )


if __name__ == "__main__":
    main()


PAYER NETWORK DATA LOADER
Started at: 2026-02-04 18:24:29
⚠️  No schema file provided or file not found

LOADING DATA FROM EXCEL FILE

📊 Found 9 sheets in Excel file

✅ Providers                      → providers                      (450 rows × 30 columns)
✅ Members                        → members                        (15,000 rows × 23 columns)
✅ Referral_Network               → referral_network               (1,995 rows × 8 columns)
✅ Monthly_Metrics                → monthly_metrics                (24 rows × 15 columns)
✅ Provider_Pods                  → provider_pods                  (4 rows × 11 columns)
✅ Behavior_Opportunities         → behavior_opportunities         (12 rows × 11 columns)
✅ High_Leakage_Providers         → high_leakage_providers         (50 rows × 10 columns)
✅ High_Risk_Members              → high_risk_members              (3,744 rows × 12 columns)
✅ Data_Dictionary                → data_dictionary                (8 rows × 3 columns)

✅ Successfully loaded 9

In [5]:
import sqlite3
import pandas as pd

# Connect to database
db_path = "my_database.db"  # Update if different
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print("="*80)
print("DATABASE VERIFICATION CHECK")
print("="*80)

# 1. Check if database exists and is accessible
print("\n✅ Database connected successfully\n")

# 2. List all tables
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print(f"📋 Tables Found: {len(tables)}")
print("-"*80)

total_records = 0
for table in tables:
    table_name = table[0]
    cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
    count = cursor.fetchone()[0]
    total_records += count
    print(f"  • {table_name:30} {count:>10,} rows")

print("-"*80)
print(f"  TOTAL RECORDS: {total_records:>10,}")
print("="*80)

# 3. Verify expected tables exist
expected_tables = [
    'providers', 'members', 'referral_network', 'monthly_metrics',
    'provider_pods', 'behavior_opportunities', 'high_leakage_providers',
    'high_risk_members', 'data_dictionary'
]

print("\n🔍 Checking for Expected Tables:")
print("-"*80)
existing_tables = [t[0] for t in tables]

for expected in expected_tables:
    if expected in existing_tables:
        print(f"  ✅ {expected}")
    else:
        print(f"  ❌ {expected} - MISSING!")

# 4. Sample data from each table
print("\n" + "="*80)
print("📊 SAMPLE DATA FROM EACH TABLE (First 3 Rows)")
print("="*80)

for table_name in existing_tables[:3]:  # Show first 3 tables
    print(f"\n{table_name.upper()}")
    print("-"*80)
    query = f"SELECT * FROM {table_name} LIMIT 3"
    df = pd.read_sql_query(query, conn)
    print(df.to_string(index=False))
    print()

# 5. Check column names for one table (to verify snake_case conversion)
print("\n" + "="*80)
print("🔤 COLUMN NAMES CHECK (providers table)")
print("="*80)
cursor.execute("PRAGMA table_info(providers);")
columns = cursor.fetchall()
print(f"\nTotal Columns: {len(columns)}\n")
for col in columns[:10]:  # Show first 10 columns
    print(f"  • {col[1]}")
if len(columns) > 10:
    print(f"  ... and {len(columns) - 10} more columns")

# 6. Run some validation queries
print("\n" + "="*80)
print("✅ VALIDATION QUERIES")
print("="*80)

# Query 1: Check provider specialties
print("\n1. Providers by Specialty:")
print("-"*80)
query = """
SELECT specialty, COUNT(*) as count
FROM providers
GROUP BY specialty
ORDER BY count DESC
LIMIT 5
"""
result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

# Query 2: Check member risk distribution
print("\n2. Members by Risk Level:")
print("-"*80)
query = """
SELECT risk_level, COUNT(*) as count
FROM members
GROUP BY risk_level
"""
result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

# Query 3: Check date range in monthly metrics
print("\n3. Monthly Metrics Date Range:")
print("-"*80)
query = """
SELECT MIN(month) as earliest_month, 
       MAX(month) as latest_month,
       COUNT(*) as total_months
FROM monthly_metrics
"""
result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

# 7. Check for any NULL values
print("\n" + "="*80)
print("🔍 NULL VALUE CHECK (providers table)")
print("="*80)
cursor.execute("PRAGMA table_info(providers);")
columns = cursor.fetchall()
column_names = [col[1] for col in columns]

null_counts = []
for col in column_names[:5]:  # Check first 5 columns
    query = f"SELECT COUNT(*) FROM providers WHERE {col} IS NULL"
    cursor.execute(query)
    null_count = cursor.fetchone()[0]
    if null_count > 0:
        null_counts.append(f"  ⚠️  {col}: {null_count} nulls")
    else:
        null_counts.append(f"  ✅ {col}: No nulls")

for item in null_counts:
    print(item)

# 8. Final summary
print("\n" + "="*80)
print("📈 SUMMARY")
print("="*80)
print(f"✅ Database: {db_path}")
print(f"✅ Tables Loaded: {len(tables)}/{len(expected_tables)}")
print(f"✅ Total Records: {total_records:,}")
print(f"✅ Data appears to be loaded correctly!")
print("="*80)

# Close connection
conn.close()
print("\n🔒 Database connection closed")

DATABASE VERIFICATION CHECK

✅ Database connected successfully

📋 Tables Found: 11
--------------------------------------------------------------------------------
  • fnol_data                           1,000 rows
  • policy_data                         1,000 rows
  • providers                             450 rows
  • members                            15,000 rows
  • referral_network                    1,995 rows
  • monthly_metrics                        24 rows
  • provider_pods                           4 rows
  • behavior_opportunities                 12 rows
  • high_leakage_providers                 50 rows
  • high_risk_members                   3,744 rows
  • data_dictionary                         8 rows
--------------------------------------------------------------------------------
  TOTAL RECORDS:     23,287

🔍 Checking for Expected Tables:
--------------------------------------------------------------------------------
  ✅ providers
  ✅ members
  ✅ referral_network
  ✅ m